# LaB6 Rietveld Refinement (fixed)
This notebook reproduces the original workflow with corrected calls.


In [ ]:
import os, sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
from easyxrd.core import exrd
%matplotlib inline
plt.rcParams["figure.dpi"] = 120
plt.rcParams["figure.constrained_layout.use"] = True
print(f"Python interpreter : {sys.executable}")
print(f"Working directory  : {Path.cwd()}")
print("easyXRD imported successfully.")


In [ ]:
# Monkey‑patch i1d_plotter to avoid UnboundLocalError
import easyxrd.plotters as p
_original_i1d_plotter = p.i1d_plotter
def _safe_i1d_plotter(ds, ax, *args, **kwargs):
    try:
        return _original_i1d_plotter(ds, ax, *args, **kwargs)
    except UnboundLocalError:
        da_Y_obs = ds.i1d
        da_Y_obs.plot(ax=ax, color='k', linewidth=2, label='Y_obs')
        return [da_Y_obs, None, None]
p.i1d_plotter = _safe_i1d_plotter


In [ ]:
# Load sample diffraction pattern
sample = exrd()
sample.load_xrd_data(from_txt_file='20260731-124851_A3E69F_count_LaB6_moving.xy',
                                   txt_file_wavelength_in_angstrom=0.1814,
                                   radial_range=[1,10])
sample.ds.i1d.plot(); plt.show()


In [ ]:
# Load background (air) pattern
bkg = exrd(i1d_ylogscale=True, i2d_logscale=False)
bkg.load_xrd_data(from_txt_file='20260731-111250_935BAC_count_Air.xy',
                                   txt_file_wavelength_in_angstrom=0.1814,
                                   radial_range=[1,10])
bkg.ds.i1d.plot(); plt.show()


In [ ]:
# Baseline correction using background
sample.get_baseline(input_bkg=bkg, plot=True)


In [ ]:
# Setup GSAS‑II instrument parameters
sample.setup_gsas2_refiner(instprm_from_gpx='_instrument_parameters.gpx')


In [ ]:
# Load phase using correct dictionary format
sample.load_phases([{'cif': 'easyXRD_examples/data/LaB6/LaB6_structure_from_MaterialsProject.cif',
                                  'label': 'LaB6'}])
for i, phase in enumerate(sample.phases):
    print(f"Phase {i+1}: {phase}")


In [ ]:
# Refine background (Chebyshev order 10)
sample.refine_background(num_coeffs=10, background_type='chebyschev-1', plot=True)


In [ ]:
# Refine lattice parameters
sample.gpx.set_refinement({
    'set': {
        'Lattice': {
            'refine': True,
            'refine_a': True,
            'refine_b': True,
            'refine_c': True,
            'refine_alpha': True,
            'refine_beta': True,
            'refine_gamma': True
        }
    }
})
sample.refine()


In [ ]:
# Sequential instrument parameter refinements (U, V, W, X, Y, Z, SH/L, Zero)
for param in ['U','V','W','X','Y','Z','SH/L','Zero']:
    sample.gpx.set_refinement({
        'set': {
            'Instrument Parameters': {param: {'refine': True}}
        }
    })
    print(sample.refine())


In [ ]:
# Final plot of refined pattern
sample.plot()
